# VirConvNet — Multimodal LiDAR + Camera Semantic Segmentation

Implements the core ideas of **VirConvNet** (Wu et al., CVPR 2023, arXiv 2303.02314)
adapted for per-point semantic segmentation instead of 3-D bounding-box detection.

## Key contributions from VirConvNet applied here
| Paper concept | Implementation |
|---|---|
| Virtual point generation | Back-project image foreground pixels to 3-D using neighbouring LiDAR depths |
| Stochastic Voxel Discard | Random dropout of virtual points during training (replaces sparse-voxel StVD) |
| Joint 3-D / 2-D feature encoding (NRConv) | Image-feature augmentation of every real LiDAR point via camera projection |

## Data layout on Google Drive
```
training_data/
  lidar/  {train,val}/{scans,labels}/{NNNNNN.npy}
  images/ {train,val}/{images,masks}/{NNNNNN.png}
```
Frame indices are aligned — `000042.npy` ↔ `000042.png`.

## Architecture
```
RGB image ──► ImageBackbone ──► (32, H/4, W/4) feature map
                                        │
                    ┌───────────────────┤
                    │  Project LiDAR → image → sample features per real point
                    │  Generate virtual 3-D points from foreground pixels
                    └───────────────────┤
LiDAR cloud ─────────────────────────► Augmented cloud (real + virtual)
                                        │ each point: 4 LiDAR + 32 img + 1 flag
                                        ▼
                              VirPillarFeatureNet (37 → 64)
                                        │
                                  BEV Backbone (64 → 320)
                                        │
                              Segmentation head (320 → 4)
                                        │
                           per-point class logits via pillar indexing
```

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy opencv-python-headless

In [ ]:
# ── 2. Imports and configuration ─────────────────────────────────────────────
import os, glob, math, random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

torch.manual_seed(42)
np.random.seed(42)


class Config:
    # ── Paths ────────────────────────────────────────────────────────────────
    DATA_ROOT  = '/content/drive/MyDrive/training_data'
    LIDAR_DIR  = DATA_ROOT + '/lidar'
    IMAGE_DIR  = DATA_ROOT + '/images'
    CKPT_DIR   = DATA_ROOT + '/checkpoints/virconvnet'

    # ── Classes ──────────────────────────────────────────────────────────────
    NUM_CLASSES = 4
    CLASS_NAMES = ['background', 'shelf', 'crate', 'forklift']
    FOREGROUND  = [1, 2, 3]          # classes to use for virtual point gen

    # ── Camera intrinsics ────────────────────────────────────────────────────
    # Derived from simulation config:
    #   IMAGE_COLLECTOR_WIDTH=640, HEIGHT=480, VFOV=90°
    #   fy = H / (2·tan(45°)) = 240;  fx = 240 (square pixels)
    IMG_W, IMG_H = 640, 480
    FX = FY = 240.0
    CX, CY  = 320.0, 240.0
    # Feature-map resolution (ImageBackbone stride 4)
    FEAT_W  = IMG_W // 4   # 160
    FEAT_H  = IMG_H // 4   # 120

    # ── Pillar geometry (sensor frame: +x fwd, +y left, +z up) ───────────────
    X_MIN, X_MAX = -20.0, 20.0
    Y_MIN, Y_MAX = -20.0, 20.0
    Z_MIN, Z_MAX =  -0.5,  3.0
    VX,    VY    =  0.25,  0.25
    MAX_PTS_PER_PILLAR = 32
    MAX_PILLARS        = 8000
    BEV_H = int((X_MAX - X_MIN) / VX)   # 160
    BEV_W = int((Y_MAX - Y_MIN) / VY)   # 160

    # ── Image feature dimension ──────────────────────────────────────────────
    IMG_FEAT_DIM   = 32
    # Input feature dim to PFN:
    #   4 (lidar raw) + 3 (Δ centroid) + 2 (pillar ctr) + IMG_FEAT_DIM + 1 (is_virtual)
    PFN_IN_CH      = 4 + 3 + 2 + IMG_FEAT_DIM + 1   # = 42
    PILLAR_FEAT    = 64

    # ── Virtual points ───────────────────────────────────────────────────────
    MAX_VIRTUAL_PTS    = 1500    # max virtual pts added per scan
    VIRTUAL_DROPOUT    = 0.3     # randomly drop virtual pts during training
    MAX_DEPTH_SEARCH_PX = 12     # max feature-pixel radius for depth lookup

    # ── Training ─────────────────────────────────────────────────────────────
    BATCH_SIZE    = 2    # smaller batch — images add memory
    NUM_EPOCHS    = 60
    LR            = 1e-3
    WEIGHT_DECAY  = 1e-4
    LR_MILESTONES = [30, 50]
    MAX_PTS       = 15_000

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


cfg = Config()
os.makedirs(cfg.CKPT_DIR, exist_ok=True)
print(f'Device     : {cfg.DEVICE}')
print(f'BEV grid   : {cfg.BEV_H} × {cfg.BEV_W}')
print(f'PFN input  : {cfg.PFN_IN_CH} features/point')

In [ ]:
# ── 3. Multimodal dataset ─────────────────────────────────────────────────────
class MultimodalDataset(Dataset):
    """
    Loads synchronised LiDAR scans + RGB images + semantic masks.
    Frame indices are aligned: 000042.npy ↔ 000042.png.
    Silently skips LiDAR frames whose paired image is missing.
    """
    def __init__(self, split='train', augment=False):
        self.augment = augment and (split == 'train')
        scan_dir = os.path.join(cfg.LIDAR_DIR, split, 'scans')
        lbl_dir  = os.path.join(cfg.LIDAR_DIR, split, 'labels')
        img_dir  = os.path.join(cfg.IMAGE_DIR, split, 'images')
        msk_dir  = os.path.join(cfg.IMAGE_DIR, split, 'masks')

        all_scans = sorted(glob.glob(os.path.join(scan_dir, '*.npy')))
        self.scans, self.labels, self.images, self.masks = [], [], [], []
        for sf in all_scans:
            stem = os.path.splitext(os.path.basename(sf))[0]
            img_f = os.path.join(img_dir, stem + '.png')
            msk_f = os.path.join(msk_dir, stem + '.png')
            if os.path.exists(img_f) and os.path.exists(msk_f):
                self.scans.append(sf)
                self.labels.append(os.path.join(lbl_dir, stem + '.npy'))
                self.images.append(img_f)
                self.masks.append(msk_f)
        print(f'[{split}] {len(self.scans)} paired LiDAR+image frames')

    def __len__(self): return len(self.scans)

    def __getitem__(self, idx):
        pts = np.load(self.scans[idx]).astype(np.float32)    # (N, 4)
        lbl = np.load(self.labels[idx]).astype(np.int64)     # (N,)
        # OpenCV loads BGR; convert to RGB and normalise to [0,1]
        img = cv2.cvtColor(cv2.imread(self.images[idx]), cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0                  # (H, W, 3)
        msk = cv2.imread(self.masks[idx], cv2.IMREAD_GRAYSCALE).astype(np.int64)  # (H, W)

        # Spatial crop
        keep = (
            (pts[:, 0] >= cfg.X_MIN) & (pts[:, 0] < cfg.X_MAX) &
            (pts[:, 1] >= cfg.Y_MIN) & (pts[:, 1] < cfg.Y_MAX) &
            (pts[:, 2] >= cfg.Z_MIN) & (pts[:, 2] < cfg.Z_MAX)
        )
        pts, lbl = pts[keep], lbl[keep]

        # Random subsample
        if len(pts) > cfg.MAX_PTS:
            sel = np.random.choice(len(pts), cfg.MAX_PTS, replace=False)
            pts, lbl = pts[sel], lbl[sel]

        # Augmentation: yaw rotation (applied consistently to LiDAR; image unchanged)
        if self.augment and len(pts) > 0:
            angle = np.random.uniform(-np.pi / 6, np.pi / 6)  # ±30° only (camera is forward-facing)
            c, s  = np.cos(angle), np.sin(angle)
            xy    = pts[:, :2] @ np.array([[c, s], [-s, c]], np.float32)
            pts   = np.concatenate([xy, pts[:, 2:]], axis=1)

        # Convert image to (3, H, W) tensor
        img_t = torch.from_numpy(img.transpose(2, 0, 1))   # (3, H, W)

        return pts, lbl, img_t, msk


def collate_mm(batch):
    pts_list = [b[0] for b in batch]
    lbl_list = [b[1] for b in batch]
    img_t    = torch.stack([b[2] for b in batch])   # (B, 3, H, W)
    msk_list = [b[3] for b in batch]
    return pts_list, lbl_list, img_t, msk_list


train_ds = MultimodalDataset('train', augment=True)
val_ds   = MultimodalDataset('val',   augment=False)
# num_workers=0 required on Colab — forked workers deadlock with CUDA init
train_dl = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                      shuffle=True,  collate_fn=collate_mm, num_workers=0,
                      pin_memory=(cfg.DEVICE == 'cuda'))
val_dl   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE,
                      shuffle=False, collate_fn=collate_mm, num_workers=0,
                      pin_memory=(cfg.DEVICE == 'cuda'))
print(f'Train batches: {len(train_dl)}   Val batches: {len(val_dl)}')

In [ ]:
# ── 4. Camera projection utilities ────────────────────────────────────────────
def lidar_to_image(pts):
    """
    Project LiDAR points (sensor frame: +x fwd, +y left, +z up)
    onto the image plane (640 × 480).

    Camera frame (OpenCV convention): +x right, +y down, +z forward.
    Transform:
        cam_x = -lidar_y
        cam_y = -lidar_z
        cam_z =  lidar_x   (depth)

    Projection:
        u = fx * cam_x / cam_z + cx
        v = fy * cam_y / cam_z + cy

    Parameters
    ----------
    pts : (N, 4) float32  LiDAR points

    Returns
    -------
    u, v  : (N,) float32  pixel coordinates
    depth : (N,) float32  cam_z (camera depth)
    valid : (N,) bool     point is in front of camera and within image bounds
    """
    lx, ly, lz = pts[:, 0], pts[:, 1], pts[:, 2]
    depth    = lx                       # cam_z = lidar_x
    in_front = depth > 0.1

    with np.errstate(divide='ignore', invalid='ignore'):
        u = np.where(in_front, cfg.FX * (-ly) / depth + cfg.CX, -1.0)
        v = np.where(in_front, cfg.FY * (-lz) / depth + cfg.CY, -1.0)

    valid = in_front & (u >= 0) & (u < cfg.IMG_W) & (v >= 0) & (v < cfg.IMG_H)
    return u.astype(np.float32), v.astype(np.float32), depth.astype(np.float32), valid


def sample_feat_at_pixels(feat_map_np, u_img, v_img):
    """
    Bilinear-sample a (C, fH, fW) feature map at original-image pixel positions.
    Scales u,v from image space to feature-map space (stride = 4).

    Returns (N, C) float32.
    """
    C, fH, fW = feat_map_np.shape
    # Feature-map coordinates
    u_f = u_img / 4.0
    v_f = v_img / 4.0
    # Bilinear interpolation using integer neighbours
    u0  = np.clip(np.floor(u_f).astype(int), 0, fW - 1)
    v0  = np.clip(np.floor(v_f).astype(int), 0, fH - 1)
    u1  = np.clip(u0 + 1, 0, fW - 1)
    v1  = np.clip(v0 + 1, 0, fH - 1)
    wu  = (u_f - u0).astype(np.float32).clip(0, 1)
    wv  = (v_f - v0).astype(np.float32).clip(0, 1)
    # (C, N) → (N, C)
    out = (
        feat_map_np[:, v0, u0] * (1 - wu) * (1 - wv) +
        feat_map_np[:, v0, u1] * wu       * (1 - wv) +
        feat_map_np[:, v1, u0] * (1 - wu) * wv       +
        feat_map_np[:, v1, u1] * wu       * wv
    ).T   # (N, C)
    return out

In [ ]:
# ── 5. Image backbone ─────────────────────────────────────────────────────────
class ImageBackbone(nn.Module):
    """
    Lightweight encoder that extracts dense image features at 1/4 resolution.

    Inputs : (B, 3, 480, 640)  normalised RGB
    Outputs: (B, 32, 120, 160) feature map

    The relatively low stride (4) preserves spatial detail for precise
    LiDAR-pixel correspondence.
    """
    def __init__(self, out_ch=32):
        super().__init__()
        self.enc = nn.Sequential(
            # stride-2: 240 × 320
            nn.Conv2d(3,  32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            # stride-2: 120 × 160
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.proj = nn.Conv2d(64, out_ch, 1, bias=False)
        self.out_ch = out_ch

    def forward(self, img):   # (B, 3, H, W)
        return self.proj(self.enc(img))   # (B, out_ch, H/4, W/4)

In [ ]:
# ── 6. Virtual point generation ───────────────────────────────────────────────
def generate_virtual_points(pts, mask_np, feat_map_np, training=False):
    """
    Generate virtual 3-D points from image foreground regions not covered
    by real LiDAR returns — the central idea of VirConvNet.

    Algorithm
    ---------
    1. Project real LiDAR points to image → sparse depth buffer at fH×fW.
    2. Find foreground pixels (mask ≠ 0) with empty depth buffer cells.
    3. For each uncovered foreground pixel, find the nearest occupied depth
       cell (within MAX_DEPTH_SEARCH_PX).  Use that depth for back-projection.
    4. Back-project pixel centre to 3-D sensor frame.
    5. Sample image features at the virtual pixel.
    6. Optionally apply Stochastic Virtual Dropout (VirConvNet §3.2).

    Parameters
    ----------
    pts        : (N, 4) float32  real LiDAR points (sensor frame)
    mask_np    : (H, W) int64   semantic mask (0 = background)
    feat_map_np: (C, fH, fW) float32  image features (stride-4)
    training   : bool  apply stochastic dropout to virtual points

    Returns
    -------
    virt_pts  : (M, 4)  float32  [x, y, z, intensity=0.5]  sensor frame
    virt_feat : (M, C)  float32  image features at virtual pixels
    """
    C, fH, fW = feat_map_np.shape

    if len(pts) == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0, C), np.float32)

    # 1. Project LiDAR → sparse depth buffer (feature-map resolution)
    u_img, v_img, depth, valid = lidar_to_image(pts)
    depth_buf = np.zeros((fH, fW), np.float32)
    if valid.any():
        u_f = np.clip((u_img[valid] / 4).astype(int), 0, fW - 1)
        v_f = np.clip((v_img[valid] / 4).astype(int), 0, fH - 1)
        # Keep nearest depth per cell
        for ui, vi, d in zip(u_f, v_f, depth[valid]):
            if depth_buf[vi, ui] == 0 or d < depth_buf[vi, ui]:
                depth_buf[vi, ui] = d

    # 2. Find foreground pixels (at feat-map resolution) with no LiDAR coverage
    # Downsample mask to feature-map size
    msk_small = mask_np[::4, ::4][:fH, :fW]   # rough nearest-neighbour
    fg_mask   = (msk_small > 0) & (depth_buf == 0)
    fg_rows, fg_cols = np.where(fg_mask)

    if len(fg_rows) == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0, C), np.float32)

    # 3. Find nearest occupied depth cell for each foreground pixel
    occ_rows, occ_cols = np.where(depth_buf > 0)
    if len(occ_rows) == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0, C), np.float32)

    occ_pts_2d  = np.stack([occ_rows, occ_cols], axis=1).astype(np.float32)
    fg_pts_2d   = np.stack([fg_rows,  fg_cols],  axis=1).astype(np.float32)
    tree        = cKDTree(occ_pts_2d)
    dists, idxs = tree.query(fg_pts_2d, k=1, workers=-1)

    # Keep only foreground pixels close enough to a LiDAR depth
    near = dists <= cfg.MAX_DEPTH_SEARCH_PX
    fg_rows  = fg_rows[near];  fg_cols  = fg_cols[near]
    ref_depth = depth_buf[occ_rows[idxs[near]], occ_cols[idxs[near]]]

    if len(fg_rows) == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0, C), np.float32)

    # Limit count
    if len(fg_rows) > cfg.MAX_VIRTUAL_PTS:
        sel = np.random.choice(len(fg_rows), cfg.MAX_VIRTUAL_PTS, replace=False)
        fg_rows, fg_cols, ref_depth = fg_rows[sel], fg_cols[sel], ref_depth[sel]

    # Stochastic Virtual Dropout (VirConvNet §3.2)
    if training and cfg.VIRTUAL_DROPOUT > 0:
        keep = np.random.rand(len(fg_rows)) > cfg.VIRTUAL_DROPOUT
        fg_rows, fg_cols, ref_depth = fg_rows[keep], fg_cols[keep], ref_depth[keep]

    if len(fg_rows) == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0, C), np.float32)

    # 4. Back-project to 3-D sensor frame
    # Feature-map pixel → original image pixel centre
    u_virt = fg_cols * 4 + 1.5    # centred on 4-px block
    v_virt = fg_rows * 4 + 1.5
    cam_z  = ref_depth
    # cam_x = (u - cx) * cam_z / fx;  cam_y = (v - cy) * cam_z / fy
    cam_x  = (u_virt - cfg.CX) * cam_z / cfg.FX
    cam_y  = (v_virt - cfg.CY) * cam_z / cfg.FY
    # sensor frame: lx = cam_z, ly = -cam_x, lz = -cam_y
    lx_v   = cam_z.astype(np.float32)
    ly_v   = (-cam_x).astype(np.float32)
    lz_v   = (-cam_y).astype(np.float32)

    # 5. Sample image features at virtual pixels
    virt_feat = sample_feat_at_pixels(feat_map_np,
                                       fg_cols * 4 + 1.5, fg_rows * 4 + 1.5)

    virt_pts = np.stack([lx_v, ly_v, lz_v,
                          np.full(len(lx_v), 0.5, np.float32)], axis=1)  # (M, 4)
    return virt_pts, virt_feat


# Quick smoke test
_pts_s, _lbl_s, _img_s, _msk_s = train_ds[0]
_img_bb = ImageBackbone(out_ch=cfg.IMG_FEAT_DIM)
with torch.no_grad():
    _fm = _img_bb(_img_s.unsqueeze(0)).squeeze(0).numpy()  # (32, 120, 160)
print(f'Feature map shape: {_fm.shape}')
_vp, _vf = generate_virtual_points(_pts_s, _msk_s, _fm, training=False)
print(f'Virtual points: {len(_vp)}  (feat shape {_vf.shape})')

In [ ]:
# ── 7. VirConvNet-style pillarization ────────────────────────────────────────
# Extended pillar feature dim:
#   4 (x,y,z,r)  +  3 (Δ centroid)  +  2 (pillar ctr)  +  32 (img)  +  1 (virtual flag)
#                                                                      = 42

def augment_and_pillarize(pts_list, lbl_list, feat_maps_np,
                           msk_list, training=False):
    """
    For each sample in the batch:
      a) Project real LiDAR → sample image features.
      b) Generate virtual points from image foreground.
      c) Concatenate real+virtual into a single augmented cloud.
      d) Pillarize with the extended 42-d feature vector.

    Returns
    -------
    pillars  : (B, P, K, PFN_IN_CH)
    coords   : (B, P, 2)
    n_pil    : (B,)
    pt2pil   : list of (N_real_b,)  — only for real LiDAR points
    lbl_list : passed through (labels exist only for real points)
    """
    B   = len(pts_list)
    P   = cfg.MAX_PILLARS
    K   = cfg.MAX_PTS_PER_PILLAR
    FC  = cfg.PFN_IN_CH           # 42
    C   = cfg.IMG_FEAT_DIM        # 32

    pillars = np.zeros((B, P, K, FC), np.float32)
    coords  = np.zeros((B, P, 2),    np.int32)
    n_pil   = np.zeros(B,            np.int32)
    pt2pil  = []

    for b in range(B):
        pts      = pts_list[b]       # (N, 4) real LiDAR
        feat_map = feat_maps_np[b]   # (C, fH, fW)
        mask_np  = msk_list[b]       # (H, W)
        N_real   = len(pts)

        # ── (a) Image features for real LiDAR points ─────────────────────
        img_feat_real = np.zeros((N_real, C), np.float32)
        if N_real > 0:
            u, v, _, vis = lidar_to_image(pts)
            if vis.any():
                img_feat_real[vis] = sample_feat_at_pixels(
                    feat_map, u[vis], v[vis])

        # ── (b) Virtual points ────────────────────────────────────────────
        virt_pts, virt_feat = generate_virtual_points(
            pts, mask_np, feat_map, training=training)
        N_virt = len(virt_pts)

        # ── (c) Build augmented combined cloud ────────────────────────────
        # Real: (N_real, 4 + C + 1) with is_virtual = 0
        real_aug = np.concatenate(
            [pts, img_feat_real, np.zeros((N_real, 1), np.float32)], axis=1)
        # Virtual: (N_virt, 4 + C + 1) with is_virtual = 1
        if N_virt > 0:
            virt_aug = np.concatenate(
                [virt_pts, virt_feat, np.ones((N_virt, 1), np.float32)], axis=1)
            combined = np.concatenate([real_aug, virt_aug], axis=0)
        else:
            combined = real_aug   # (N_real, 37)

        # combined[:, :4] = x, y, z, intensity
        # combined[:, 4:4+C] = image features
        # combined[:, 4+C] = is_virtual

        total_pts = len(combined)   # N_real + N_virt
        if total_pts == 0:
            pt2pil.append(np.empty(0, np.int32))
            continue

        xi = np.clip(np.floor((combined[:, 0] - cfg.X_MIN) / cfg.VX).astype(np.int32),
                     0, cfg.BEV_H - 1)
        yi = np.clip(np.floor((combined[:, 1] - cfg.Y_MIN) / cfg.VY).astype(np.int32),
                     0, cfg.BEV_W - 1)
        key = xi.astype(np.int64) * cfg.BEV_W + yi

        order  = np.argsort(key, kind='mergesort')
        key_s  = key[order]
        pts_s  = combined[order]
        bounds = np.where(np.diff(key_s, prepend=key_s[0] - 1) != 0)[0]
        ukeys  = key_s[bounds]
        n_keep = min(len(ukeys), P)

        # pt2pil tracks REAL points only (first N_real in combined)
        p2p_real = np.full(N_real, -1, np.int32)

        for pi in range(n_keep):
            s   = bounds[pi]
            e   = bounds[pi + 1] if pi + 1 < len(bounds) else total_pts
            n   = min(e - s, K)
            orig_idx = order[s: s + n]
            # Map only real-point indices
            real_mask = orig_idx < N_real
            p2p_real[orig_idx[real_mask]] = pi

            pts_pi  = pts_s[s: s + n]           # (n, 37)
            mean    = pts_pi[:, :3].mean(0)     # x,y,z centroid
            xi_p    = int(ukeys[pi]) // cfg.BEV_W
            yi_p    = int(ukeys[pi]) %  cfg.BEV_W
            xp      = xi_p * cfg.VX + cfg.X_MIN + cfg.VX / 2
            yp      = yi_p * cfg.VY + cfg.Y_MIN + cfg.VY / 2

            # Build 42-d feature vector
            # [x,y,z,r | Δx,Δy,Δz | xp,yp | img_feat(32) | is_virtual]
            feat = np.zeros((n, FC), np.float32)
            feat[:, :4]          = pts_pi[:, :4]                    # x,y,z,r
            feat[:, 4:7]         = pts_pi[:, :3] - mean             # Δx,Δy,Δz
            feat[:, 7]           = xp
            feat[:, 8]           = yp
            feat[:, 9:9 + C]     = pts_pi[:, 4:4 + C]              # img features
            feat[:, 9 + C]       = pts_pi[:, 4 + C]                # is_virtual

            pillars[b, pi, :n]   = feat
            coords[b, pi]        = [xi_p, yi_p]

        n_pil[b] = n_keep
        pt2pil.append(p2p_real)

    return (
        torch.from_numpy(pillars),
        torch.from_numpy(coords),
        n_pil,
        pt2pil,
    )

In [ ]:
# ── 8. VirConvNet model ───────────────────────────────────────────────────────
class VirPillarFeatureNet(nn.Module):
    """PFN with extended input dimension to accept image features."""
    def __init__(self, in_ch, out_ch=64):
        super().__init__()
        self.lin = nn.Linear(in_ch, out_ch, bias=False)
        self.bn  = nn.BatchNorm1d(out_ch)

    def forward(self, pillars):
        BP, K, C = pillars.shape
        x = self.lin(pillars.reshape(BP * K, C))
        x = F.relu(self.bn(x), inplace=True)
        return x.view(BP, K, -1).max(dim=1)[0]


class _ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, depth=4):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
                  nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        for _ in range(depth - 1):
            layers += [nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


class BEVBackbone(nn.Module):
    def __init__(self, in_ch=64):
        super().__init__()
        self.b1  = _ConvBlock(in_ch, 64,  stride=1, depth=4)
        self.b2  = _ConvBlock(64,   128,  stride=2, depth=6)
        self.b3  = _ConvBlock(128,  256,  stride=2, depth=6)
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 2, stride=2, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=4, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.out_ch = 64 + 128 + 128

    def forward(self, x):
        f1 = self.b1(x);  f2 = self.b2(f1);  f3 = self.b3(f2)
        return torch.cat([f1, self.up2(f2), self.up3(f3)], dim=1)


class VirConvNetSeg(nn.Module):
    """
    VirConvNet for semantic segmentation.

    The image backbone runs on the full batch in one GPU call;
    virtual point generation and pillarization happen on CPU per sample.
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.img_bb   = ImageBackbone(out_ch=cfg.IMG_FEAT_DIM)
        self.pfn      = VirPillarFeatureNet(in_ch=cfg.PFN_IN_CH, out_ch=cfg.PILLAR_FEAT)
        self.backbone = BEVBackbone(in_ch=cfg.PILLAR_FEAT)
        self.head     = nn.Sequential(
            nn.Conv2d(self.backbone.out_ch, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(128, num_classes, 1),
        )
        self.num_classes = num_classes

    def encode_images(self, imgs):   # (B, 3, H, W) → (B, C, fH, fW)
        return self.img_bb(imgs)

    def forward_pillars(self, pillars, coords, n_pil):
        """
        pillars : (B, P, K, PFN_IN_CH)  on device
        coords  : (B, P, 2)
        n_pil   : (B,)
        Returns: logits_bev (B, C, H, W)
        """
        B, P, K, C = pillars.shape
        H, W = cfg.BEV_H, cfg.BEV_W

        pf  = self.pfn(pillars.view(B * P, K, C)).view(B, P, cfg.PILLAR_FEAT)
        bev = pillars.new_zeros(B, cfg.PILLAR_FEAT, H, W)
        for b in range(B):
            n  = int(n_pil[b])
            xi = coords[b, :n, 0].long()
            yi = coords[b, :n, 1].long()
            bev[b, :, xi, yi] = pf[b, :n].T

        feat = self.backbone(bev)
        return self.head(feat)


# Verify shapes
_model = VirConvNetSeg().to(cfg.DEVICE)
_dummy_img = torch.zeros(1, 3, cfg.IMG_H, cfg.IMG_W).to(cfg.DEVICE)
with torch.no_grad():
    _fm = _model.encode_images(_dummy_img)
print(f'Feature map: {tuple(_fm.shape)}  expected (1, {cfg.IMG_FEAT_DIM}, {cfg.FEAT_H}, {cfg.FEAT_W})')
total_params = sum(p.numel() for p in _model.parameters())
print(f'Parameters : {total_params:,}')
del _model, _dummy_img, _fm

In [ ]:
# ── 9. Loss and metrics ───────────────────────────────────────────────────────
def estimate_class_weights(dataset, n_sample=300):
    counts = np.zeros(cfg.NUM_CLASSES, np.int64)
    for i in np.random.choice(len(dataset), min(n_sample, len(dataset)), replace=False):
        _, lbl, _, _ = dataset[i]
        for c in range(cfg.NUM_CLASSES):
            counts[c] += int((lbl == c).sum())
    counts  = np.maximum(counts, 1)
    w = 1.0 / counts.astype(np.float64)
    w = w / w.sum() * cfg.NUM_CLASSES
    return torch.tensor(w, dtype=torch.float32)


print('Estimating class weights…')
class_weights = estimate_class_weights(train_ds).to(cfg.DEVICE)
print('Weights:', dict(zip(cfg.CLASS_NAMES, class_weights.cpu().numpy().round(3))))
criterion = nn.CrossEntropyLoss(weight=class_weights)


def per_class_iou(pred_np, true_np):
    iou = []
    for c in range(cfg.NUM_CLASSES):
        tp = int(((pred_np == c) & (true_np == c)).sum())
        fp = int(((pred_np == c) & (true_np != c)).sum())
        fn = int(((pred_np != c) & (true_np == c)).sum())
        d  = tp + fp + fn
        iou.append(tp / d if d > 0 else float('nan'))
    return iou

In [ ]:
# ── 10. Train / evaluate functions ────────────────────────────────────────────
def gather_point_logits(logits_bev, coords_np, n_pil, pt2pil, lbl_list=None):
    """Index BEV logits at each real point's pillar coordinate."""
    all_logits, all_labels = [], []
    B = logits_bev.shape[0]
    for b in range(B):
        p2p   = pt2pil[b]        # (N_real_b,)  -1 = dropped
        valid = p2p >= 0
        if not valid.any():
            continue
        xi = torch.from_numpy(coords_np[b, p2p[valid], 0].astype(np.int64)).to(logits_bev.device)
        yi = torch.from_numpy(coords_np[b, p2p[valid], 1].astype(np.int64)).to(logits_bev.device)
        all_logits.append(logits_bev[b, :, xi, yi].T)
        if lbl_list is not None:
            all_labels.append(
                torch.from_numpy(lbl_list[b][valid]).to(logits_bev.device))
    if not all_logits:
        return None, None
    return torch.cat(all_logits), (torch.cat(all_labels) if all_labels else None)


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for pts_list, lbl_list, imgs, msk_list in loader:
        imgs = imgs.to(cfg.DEVICE)

        # Image features for full batch — one GPU call
        with torch.no_grad():             # freeze img_bb gradient if desired:
            feat_maps = model.encode_images(imgs)   # (B, C, fH, fW)
        feat_maps_np = feat_maps.cpu().numpy()

        # Pillarize on CPU (includes virtual point generation)
        pil, coo, n_pil, p2p = augment_and_pillarize(
            pts_list, lbl_list, feat_maps_np, msk_list, training=True)
        pil = pil.to(cfg.DEVICE)
        coo = coo.to(cfg.DEVICE)

        optimizer.zero_grad()
        logits_bev        = model.forward_pillars(pil, coo, n_pil)
        logits_pt, lbl_pt = gather_point_logits(
            logits_bev, coo.cpu().numpy(), n_pil, p2p, lbl_list)
        if logits_pt is None:
            continue
        loss = criterion(logits_pt, lbl_pt)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 35.0)
        optimizer.step()
        total_loss += loss.item(); n += 1
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_pred, all_true = [], []
    for pts_list, lbl_list, imgs, msk_list in loader:
        imgs = imgs.to(cfg.DEVICE)
        feat_maps    = model.encode_images(imgs)
        feat_maps_np = feat_maps.cpu().numpy()
        pil, coo, n_pil, p2p = augment_and_pillarize(
            pts_list, lbl_list, feat_maps_np, msk_list, training=False)
        pil = pil.to(cfg.DEVICE)
        coo = coo.to(cfg.DEVICE)
        logits_bev        = model.forward_pillars(pil, coo, n_pil)
        logits_pt, lbl_pt = gather_point_logits(
            logits_bev, coo.cpu().numpy(), n_pil, p2p, lbl_list)
        if logits_pt is None:
            continue
        all_pred.append(logits_pt.argmax(1).cpu().numpy())
        all_true.append(lbl_pt.cpu().numpy())
    if not all_pred:
        return [float('nan')] * cfg.NUM_CLASSES, float('nan')
    pred_np = np.concatenate(all_pred)
    true_np = np.concatenate(all_true)
    iou     = per_class_iou(pred_np, true_np)
    valid   = [v for v in iou if not math.isnan(v)]
    miou    = sum(valid) / len(valid) if valid else float('nan')
    return iou, miou

In [ ]:
# ── 11. Training loop ─────────────────────────────────────────────────────────
model     = VirConvNetSeg(num_classes=cfg.NUM_CLASSES).to(cfg.DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR,
                              weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=cfg.LR_MILESTONES, gamma=0.1)

history   = {'loss': [], 'miou': []}
best_miou = 0.0

for epoch in range(1, cfg.NUM_EPOCHS + 1):
    loss = train_epoch(model, train_dl, optimizer)
    scheduler.step()
    history['loss'].append(loss)

    if epoch % 5 == 0 or epoch == 1:
        iou_list, miou = evaluate(model, val_dl)
        history['miou'].append((epoch, miou))

        iou_str = '  '.join(
            f'{cfg.CLASS_NAMES[c]}={v:.3f}' if not math.isnan(v) else f'{cfg.CLASS_NAMES[c]}=--'
            for c, v in enumerate(iou_list))
        print(f'Ep {epoch:3d}/{cfg.NUM_EPOCHS}  loss={loss:.4f}  '
              f'mIoU={miou:.4f}  [{iou_str}]')

        if miou > best_miou:
            best_miou = miou
            torch.save({'epoch': epoch,
                        'state_dict': model.state_dict(),
                        'miou': miou, 'iou': iou_list},
                       os.path.join(cfg.CKPT_DIR, 'best.pt'))
            print(f'  ✓ best checkpoint saved (mIoU={miou:.4f})')

        if epoch % 10 == 0:
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'scheduler': scheduler.state_dict()},
                       os.path.join(cfg.CKPT_DIR, f'epoch_{epoch:03d}.pt'))
            print(f'  ✓ periodic checkpoint saved (epoch {epoch})')

print(f'\nDone.  Best val mIoU = {best_miou:.4f}')

In [ ]:
# ── 12. Evaluation and visualisation ─────────────────────────────────────────
ckpt = torch.load(os.path.join(cfg.CKPT_DIR, 'best.pt'), map_location=cfg.DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f"Best checkpoint — epoch {ckpt['epoch']}  mIoU={ckpt['miou']:.4f}")

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['loss'])
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training loss'); axes[0].grid(True)

ep_vals  = [e for e, _ in history['miou']]
miou_vals = [m for _, m in history['miou']]
axes[1].plot(ep_vals, miou_vals, 'g-o', ms=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Validation mIoU'); axes[1].grid(True)
plt.tight_layout(); plt.show()

# Final IoU table
iou_list, miou = evaluate(model, val_dl)
print('\n── Validation IoU ──────────────────────')
print(f'{"Class":<14} {"IoU":>8}')
print('─' * 24)
for c, v in enumerate(iou_list):
    s = f'{v:.4f}' if not math.isnan(v) else '    --'
    print(f'{cfg.CLASS_NAMES[c]:<14} {s:>8}')
print(f'{"mIoU":<14} {miou:.4f}')

# Qualitative BEV plot + camera RGB
model.eval()
batch = next(iter(val_dl))
pts_list_v, lbl_list_v, imgs_v, msk_list_v = batch
imgs_v = imgs_v.to(cfg.DEVICE)
with torch.no_grad():
    fm_v = model.encode_images(imgs_v)
fm_np = fm_v.cpu().numpy()
pil_v, coo_v, n_pv, p2p_v = augment_and_pillarize(
    pts_list_v, lbl_list_v, fm_np, msk_list_v, training=False)
with torch.no_grad():
    logits_v = model.forward_pillars(pil_v.to(cfg.DEVICE), coo_v.to(cfg.DEVICE), n_pv)

COLORS = np.array([[0.45,0.45,0.45], [0.20,0.70,0.20],
                    [0.90,0.15,0.15], [0.10,0.40,0.90]])
b = 0
pts   = pts_list_v[b]
true  = lbl_list_v[b]
valid = p2p_v[b] >= 0
pts_v  = pts[valid]
true_v = true[valid]
xi_b   = coo_v[b, p2p_v[b][valid], 0].astype(np.int64)
yi_b   = coo_v[b, p2p_v[b][valid], 1].astype(np.int64)
pred_v = logits_v[b, :,
                  torch.from_numpy(xi_b).to(cfg.DEVICE),
                  torch.from_numpy(yi_b).to(cfg.DEVICE)
                 ].argmax(0).cpu().numpy()

fig = plt.figure(figsize=(18, 6))
ax0 = fig.add_subplot(1, 3, 1)
ax0.imshow(imgs_v[b].permute(1, 2, 0).cpu().numpy())
ax0.set_title('Camera RGB'); ax0.axis('off')

for ax, labels, title in zip(
        [fig.add_subplot(1,3,2), fig.add_subplot(1,3,3)],
        [true_v, pred_v], ['Ground Truth', 'VirConvNet Prediction']):
    c = COLORS[np.clip(labels, 0, 3)]
    ax.scatter(pts_v[:, 1], pts_v[:, 0], c=c, s=1.0, alpha=0.85, rasterized=True)
    ax.set_aspect('equal'); ax.set_title(title)
    ax.set_xlabel('Y — left (m)'); ax.set_ylabel('X — forward (m)')
    ax.set_xlim(cfg.Y_MIN, cfg.Y_MAX); ax.set_ylim(cfg.X_MIN, cfg.X_MAX)
    legend = [Patch(color=COLORS[c], label=cfg.CLASS_NAMES[c])
               for c in range(cfg.NUM_CLASSES)]
    ax.legend(handles=legend, loc='upper right')

plt.suptitle("VirConvNet Multimodal Segmentation — BEV", fontsize=14)
plt.tight_layout(); plt.show()

# Show virtual point coverage on one scan
vp, _ = generate_virtual_points(pts, msk_list_v[b], fm_np[b], training=False)
print(f'Virtual points added: {len(vp)}  (real: {len(pts)})')
if len(vp) > 0:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(pts[:, 1],  pts[:, 0],  s=0.8, c='steelblue',  alpha=0.5, label='Real LiDAR')
    ax.scatter(vp[:, 1],   vp[:, 0],   s=0.8, c='orange',     alpha=0.5, label='Virtual pts')
    ax.set_aspect('equal'); ax.set_title('Real vs Virtual points')
    ax.set_xlim(cfg.Y_MIN, cfg.Y_MAX); ax.set_ylim(cfg.X_MIN, cfg.X_MAX)
    ax.legend(markerscale=5); plt.tight_layout(); plt.show()